# Import

In [ ]:
import os
import bm25s
import numpy as np

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from beir.retrieval.models import SentenceBERT

# Add Data

In [ ]:
def add_data(dataset_name: str):
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset_name)
    out_dir = os.path.join(os.getcwd(), "datasets")
    data_path = util.download_and_unzip(url, out_dir)
    return data_path

# Add Models

## Add Model Dense

In [ ]:
def add_model_dense(model_name: str):
    dense_model = DRES(SentenceBERT(f"sentence-transformers/{model_name}"), batch_size=64)
    return dense_model

## Add model Sparse

In [ ]:
def add_model_sparse():
    retriever = bm25s.BM25(method="lucene", k1=1.2, b=0.75)
    return retriever

# Metrics

In [ ]:
import math

class RunDatasets:
    def __init__(self):
        self.evaluator = EvaluateRetrieval()
        self.train_retrieval_data = {}
        self.test_retrieval_data = {}

    def normalize_scores(self, results):
        """Min-Max normalization of scores for BM25 and Dense."""
        normalized = {}
        for qid, docs in results.items():
            if not docs:
                continue
            scores = list(docs.values())
            min_s, max_s = min(scores), max(scores)

            normalized[qid] = {}
            for did, score in docs.items():
                if max_s > min_s:
                    normalized[qid][did] = (score - min_s) / (max_s - min_s)
                else:
                    normalized[qid][did] = 0.5
        return normalized

    def get_hybrid_scores(self, dense_res, sparse_res, alpha):
        """alpha * Dense + (1 - alpha) * Sparse."""
        hybrid = {}
        all_qids = set(dense_res.keys()) | set(sparse_res.keys())

        for qid in all_qids:
            hybrid[qid] = {}
            q_dense = dense_res.get(qid, {})
            q_sparse = sparse_res.get(qid, {})
            all_dids = set(q_dense.keys()) | set(q_sparse.keys())

            for did in all_dids:
                s_d = q_dense.get(did, 0.0)
                s_s = q_sparse.get(did, 0.0)
                hybrid[qid][did] = float(alpha * s_d + (1.0 - alpha) * s_s)

        return hybrid

    def _process_dataset(self, ds: str, split: str):
        """Core pipeline: loads data, runs retrieval, normalizes scores."""
        print(f"\n[{ds.upper()}] Loading data (split='{split}')...")
        data_path = add_data(ds)
        corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split=split)

        corpus_ids = list(corpus.keys())
        corpus_texts = [f"{corpus[cid].get('title', '')} {corpus[cid].get('text', '')}" for cid in corpus_ids]
        query_ids = [qid for qid in queries.keys() if qid in qrels]
        query_texts = [queries[qid] for qid in query_ids]

        print(f"[{ds.upper()}] Building IDF dictionary...")
        df_dict = {}
        for text in corpus_texts:
            for token in set(text.lower().split()):
                df_dict[token] = df_dict.get(token, 0) + 1

        N = len(corpus_texts)
        idf_dict = {t: math.log(1 + (N - df + 0.5) / (df + 0.5)) for t, df in df_dict.items() if df > 0}
        vocab_set = set(df_dict.keys())

        print(f"[{ds.upper()}] Running Dense Retrieval...")
        dense_model = add_model_dense("all-MiniLM-L6-v2")
        dense_retriever = EvaluateRetrieval(dense_model, score_function="cos_sim")
        dense_raw = dense_retriever.retrieve(corpus, queries)

        print(f"[{ds.upper()}] Running Sparse Retrieval (BM25s)...")
        sparse_model = add_model_sparse()
        corpus_tokens = bm25s.tokenize(corpus_texts)
        sparse_model.index(corpus_tokens)

        query_tokens = bm25s.tokenize(query_texts)
        docs, scores = sparse_model.retrieve(query_tokens, corpus=corpus_ids, k=100)
        sparse_raw = {qid: {docs[i][j]: float(scores[i][j]) for j in range(len(docs[i]))} for i, qid in enumerate(query_ids)}

        print(f"[{ds.upper()}] Normalizing scores...")
        dense_norm = self.normalize_scores(dense_raw)
        sparse_norm = self.normalize_scores(sparse_raw)

        return {
            "queries": {qid: queries[qid] for qid in query_ids},
            "qrels": qrels,
            "dense_norm": dense_norm,
            "sparse_norm": sparse_norm,
            "idf_dict": idf_dict,
            "vocab_set": vocab_set
        }

    def run_train_datasets(self, datasets: list, split="train"):
        """Extracts and saves retrieval artifacts for training the PyTorch model."""
        for ds in datasets:
            self.train_retrieval_data[ds] = self._process_dataset(ds, split)
        print("\nTraining datasets processed and saved!")

    def run_test_datasets(self, datasets: list, split="test"):
        """Extracts artifacts, saves them for dynamic inference, and calculates static baselines."""
        alphas = np.round(np.arange(0.0, 1.05, 0.05), 2)
        results_ndcg = {ds: [] for ds in datasets}
        results_mrr = {ds: [] for ds in datasets}

        for ds in datasets:
            data = self._process_dataset(ds, split)
            self.test_retrieval_data[ds] = data # Сохраняем для инференса PyTorch

            dense_norm = data["dense_norm"]
            sparse_norm = data["sparse_norm"]
            qrels = data["qrels"]

            print(f"[{ds.upper()}] Evaluating static alphas...")
            for alpha in alphas:
                hybrid_res = self.get_hybrid_scores(dense_norm, sparse_norm, alpha)

                ndcg_dict, _, _, _ = self.evaluator.evaluate(qrels, hybrid_res, [10])
                mrr_dict = self.evaluator.evaluate_custom(qrels, hybrid_res, [10], metric="mrr")

                results_ndcg[ds].append(ndcg_dict["NDCG@10"])
                mrr_key = list(mrr_dict.keys())[0]
                results_mrr[ds].append(mrr_dict[mrr_key])

        print("\nTest benchmark completed!")
        return results_ndcg, results_mrr, alphas

In [ ]:
datasets_list = ["scifact", "fiqa", "nfcorpus", "scidocs", "msmarco"]
# datasets_list = ["scifact", "fiqa", "nfcorpus", "scidocs"]

rd = RunDatasets()
results_ndcg, results_mrr, alphas = rd.run_test_datasets(datasets_list)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i, ds in enumerate(datasets_list):
    ax1.plot(alphas, results_ndcg[ds], label=ds.upper(), color=colors[i], marker='o', markersize=4)
    max_idx = np.argmax(results_ndcg[ds])
    ax1.scatter(alphas[max_idx], results_ndcg[ds][max_idx], color=colors[i], s=100, marker='*')

ax1.set_title("NDCG@10 by Dense Weight ($\\alpha$)", fontsize=14)
ax1.set_xlabel("$\\alpha$ (0 = Pure BM25, 1 = Pure Dense)", fontsize=12)
ax1.set_ylabel("NDCG@10", fontsize=12)
ax1.set_xticks(alphas)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

for i, ds in enumerate(datasets_list):
    ax2.plot(alphas, results_mrr[ds], label=ds.upper(), color=colors[i], marker='o', markersize=4)
    max_idx = np.argmax(results_mrr[ds])
    ax2.scatter(alphas[max_idx], results_mrr[ds][max_idx], color=colors[i], s=100, marker='*')

ax2.set_title("MRR@10 by Dense Weight ($\\alpha$)", fontsize=14)
ax2.set_xlabel("$\\alpha$ (0 = Pure BM25, 1 = Pure Dense)", fontsize=12)
ax2.set_ylabel("MRR@10", fontsize=12)
ax2.set_xticks(alphas)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
# plt.savefig("hybrid_benchmark.pdf", dpi=300, format='pdf')
plt.show()

# Extract Features

In [ ]:
import nltk
import string

from nltk.corpus import stopwords
import random

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
class CalculateFeatures:
    # Initialize stopwords once at the class level for performance
    stop_words = set(stopwords.words('english'))

    @staticmethod
    def get_lexical_features(idf_dict: dict, vocab_set: set, tokens: list, q_len: int) -> list:
        """Calculates standard lexical and IDF-based features."""
        idfs = [idf_dict.get(t, 0.0) for t in tokens]

        mean_idf = float(np.mean(idfs))
        max_idf = float(np.max(idfs))
        min_idf = float(np.min(idfs))
        std_idf = float(np.std(idfs))

        # Rare token ratio (threshold > 5.0)
        rare_token_ratio = sum(1 for idf in idfs if idf > 5.0) / q_len

        # OOV ratio (Out Of Vocabulary)
        oov_ratio = sum(1 for t in tokens if t not in vocab_set) / q_len

        return [q_len, mean_idf, max_idf, min_idf, std_idf, rare_token_ratio, oov_ratio]

    @staticmethod
    def get_digit_features(chars: list, query_text: str) -> list:
        """Calculates character-level features (digits, casing, punctuation)."""
        char_len = len(chars)

        # Guard clause for empty character lists
        if char_len == 0:
            return [0.0, 0.0, 0.0]

        digit_ratio = sum(1 for c in chars if c.isdigit()) / char_len

        # Uppercase ratio - calculate with original text, not lowercased
        orig_chars = list(query_text.replace(" ", ""))
        uppercase_ratio = sum(1 for c in orig_chars if c.isupper()) / char_len

        punctuation_ratio = sum(1 for c in chars if c in string.punctuation) / char_len

        return [digit_ratio, uppercase_ratio, punctuation_ratio]

    @staticmethod
    def get_nlp_features(query_text: str, tokens: list) -> list:
        """Calculates NLP-based features like POS tags and stopword ratios."""
        nltk_tokens = nltk.word_tokenize(query_text)
        n_tokens = max(len(nltk_tokens), 1)

        # Stopword ratio
        stopword_ratio = sum(1 for t in nltk_tokens if t.lower() in CalculateFeatures.stop_words) / n_tokens

        # POS tagging (NN = Nouns, VB = Verbs, JJ = Adjectives)
        pos_tags = nltk.pos_tag(nltk_tokens)

        nouns = sum(1 for word, tag in pos_tags if tag.startswith('NN'))
        verbs = sum(1 for word, tag in pos_tags if tag.startswith('VB'))
        adjs = sum(1 for word, tag in pos_tags if tag.startswith('JJ'))

        noun_ratio = nouns / n_tokens
        verb_ratio = verbs / n_tokens
        adj_ratio = adjs / n_tokens

        # Average word length
        avg_word_len = float(np.mean([len(t) for t in tokens]))

        return [stopword_ratio, noun_ratio, verb_ratio, adj_ratio, avg_word_len]

    @staticmethod
    def extract_features(query_text: str, idf_dict: dict, vocab_set: set) -> list:
        """Orchestrates feature extraction from all categories."""
        tokens = query_text.lower().split()
        chars = list(query_text.replace(" ", ""))
        q_len = len(tokens)

        if q_len == 0:
            return [0.0] * 15

        lex_features = CalculateFeatures.get_lexical_features(idf_dict, vocab_set, tokens, q_len)
        digit_features = CalculateFeatures.get_digit_features(chars, query_text)
        nlp_features = CalculateFeatures.get_nlp_features(query_text, tokens)

        # Combine all features into a single vector
        return lex_features + digit_features + nlp_features

In [ ]:
def build_training_triplets(retrieval_data: dict, dataset_name: str, num_negatives=3):
    """
    Collects triplets (x_q, pos_scores, neg_scores) for training a PyTorch model.
    num_negatives: the number of irrelevant documents to include for every relevant document.
    """
    data = retrieval_data[dataset_name]
    queries = data["queries"]
    qrels = data["qrels"]
    dense_norm = data["dense_norm"]
    sparse_norm = data["sparse_norm"]
    idf_dict = data["idf_dict"]
    vocab_set = data["vocab_set"]

    triplets = []

    for qid, q_text in queries.items():
        if qid not in qrels:
            continue

        # 1. Get features of queries x_q
        x_q = CalculateFeatures.extract_features(q_text, idf_dict, vocab_set)

        # 2. Find positive documents
        pos_docs = [did for did, rel in qrels[qid].items() if rel > 0]
        if not pos_docs:
            continue

        # 3. Get pull of negative documents
        retrieved_docs = set(dense_norm.get(qid, {}).keys()) | set(sparse_norm.get(qid, {}).keys())
        neg_docs = list(retrieved_docs - set(pos_docs))

        # Guard clause: ensure we have exactly `num_negatives` to keep tensor shapes consistent
        if len(neg_docs) < num_negatives:
            print(f"[{dataset_name.upper()}] Warning: Insufficient negative docs for {qid}. Skipping.")
            continue

        # 4. get lists
        for pos_d in pos_docs:
            # get random negatives examples (Hard Negatives, they are from the top)
            sampled_negs = random.sample(neg_docs, min(num_negatives, len(neg_docs)))

            neg_dense_scores = [float(dense_norm.get(qid, {}).get(nd, 0.0)) for nd in sampled_negs]
            neg_sparse_scores = [float(sparse_norm.get(qid, {}).get(nd, 0.0)) for nd in sampled_negs]

            triplets.append({
                'features': x_q,
                'pos_dense': float(dense_norm.get(qid, {}).get(pos_d, 0.0)),
                'pos_sparse': float(sparse_norm.get(qid, {}).get(pos_d, 0.0)),
                'neg_dense': neg_dense_scores,   # List of K floats
                'neg_sparse': neg_sparse_scores, # List of K floats
            })

    print(f"[{dataset_name.upper()}] Generated {len(triplets)} training triplets.")
    return triplets

In [ ]:
train_datasets = ["scifact", "fiqa", "nfcorpus", "msmarco"]
# train_datasets = ["scifact", "fiqa", "nfcorpus"]
rd.run_train_datasets(train_datasets)

In [ ]:
global_triplets = []
for ds in train_datasets:
    train_triplets = build_training_triplets(rd.train_retrieval_data, dataset_name=ds, num_negatives=5)
    global_triplets.extend(train_triplets)

In [ ]:
len(global_triplets)

# Train Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
class HybridLTRDataset(Dataset):
    def __init__(self, triplets):
        """
        triplets: list of dict like:
        {global_triplets
            'features': [f1, f2, ... f15],
            'pos_dense': float, 'pos_sparse': float,
            'neg_dense': float, 'neg_sparse': float
        }
        """
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        item = self.triplets[idx]
        return {
            'x_q': torch.tensor(item['features'], dtype=torch.float32),
            's_d_pos': torch.tensor(item['pos_dense'], dtype=torch.float32),
            's_s_pos': torch.tensor(item['pos_sparse'], dtype=torch.float32),
            's_d_neg': torch.tensor(item['neg_dense'], dtype=torch.float32),
            's_s_neg': torch.tensor(item['neg_sparse'], dtype=torch.float32),
        }

class AlphaRouter(nn.Module):
    def __init__(self, input_dim=15):
        super(AlphaRouter, self).__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x_q):
        return self.net(x_q).squeeze(-1)

def pairwise_hybrid_loss(alpha, s_dense_pos, s_sparse_pos, s_dense_neg, s_sparse_neg, margin=0.1):
    hybrid_pos = alpha * s_dense_pos + (1.0 - alpha) * s_sparse_pos
    hybrid_neg = alpha * s_dense_neg + (1.0 - alpha) * s_sparse_neg
    loss = F.relu(margin - (hybrid_pos - hybrid_neg))

    return loss.mean()

def infonce_hybrid_loss(alpha, s_d_pos, s_s_pos, s_d_negs, s_s_negs, tau=0.1):
    """
    s_d_negs, s_s_negs: tensors of shape (batch_size, num_negatives)
    """
    # Calculate score for the positive document, shape: (batch_size, 1)
    hybrid_pos = (alpha * s_d_pos + (1.0 - alpha) * s_s_pos).unsqueeze(1)

    # Calculate scores for all negative documents, shape: (batch_size, num_negatives)
    alpha_expanded = alpha.unsqueeze(1)
    hybrid_negs = alpha_expanded * s_d_negs + (1.0 - alpha_expanded) * s_s_negs

    # Concatenate: the positive document is always at index 0
    logits = torch.cat([hybrid_pos, hybrid_negs], dim=1) / tau

    # Target - zero index (since the positive is always at the 0th position)
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)

    # Classical cross-entropy does all the InfoNCE magic for us
    loss = F.cross_entropy(logits, labels)

    return loss

In [ ]:
def train_router(model, dataloader, epochs=20, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in dataloader:
            optimizer.zero_grad()

            alpha = model(batch['x_q'])

            loss = infonce_hybrid_loss(
                alpha,
                batch['s_d_pos'], batch['s_s_pos'],
                batch['s_d_neg'], batch['s_s_neg'],
                tau=0.1
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / len(dataloader):.4f}")

dataloader = DataLoader(HybridLTRDataset(global_triplets), batch_size=64, shuffle=True, drop_last=True)
model = AlphaRouter()
train_router(model, dataloader)

In [ ]:
def evaluate_dynamic_router(model, retrieval_data, datasets):
    model.eval()
    evaluator = EvaluateRetrieval()

    dynamic_results = {}

    with torch.no_grad():
        for ds in datasets:
            print(f"\n[TESTING] {ds.upper()}...")
            data = retrieval_data[ds]
            queries = data["queries"]
            qrels = data["qrels"]
            dense_norm = data["dense_norm"]
            sparse_norm = data["sparse_norm"]
            idf_dict = data["idf_dict"]
            vocab_set = data["vocab_set"]

            hybrid_results = {}
            predicted_alphas = []

            for qid, q_text in queries.items():
                if qid not in qrels:
                    continue

                x_q = CalculateFeatures.extract_features(q_text, idf_dict, vocab_set)

                x_q_tensor = torch.tensor(x_q, dtype=torch.float32).unsqueeze(0)

                alpha = model(x_q_tensor).item()
                predicted_alphas.append(alpha)

                candidate_docs = set(dense_norm.get(qid, {}).keys()) | set(sparse_norm.get(qid, {}).keys())
                hybrid_results[qid] = {}

                for did in candidate_docs:
                    s_d = dense_norm.get(qid, {}).get(did, 0.0)
                    s_s = sparse_norm.get(qid, {}).get(did, 0.0)
                    hybrid_results[qid][did] = alpha * s_d + (1.0 - alpha) * s_s

            ndcg_dict, _, _, _ = evaluator.evaluate(qrels, hybrid_results, [10])
            mrr_dict = evaluator.evaluate_custom(qrels, hybrid_results, [10], metric="mrr")
            mrr_key = list(mrr_dict.keys())[0]

            mean_alpha = np.mean(predicted_alphas)
            std_alpha = np.std(predicted_alphas)

            dynamic_results[ds] = {
                "NDCG@10": ndcg_dict["NDCG@10"],
                "MRR@10": mrr_dict[mrr_key],
                "mean_alpha": mean_alpha,
                "std_alpha": std_alpha
            }

            print(f"Dynamic NDCG@10: {ndcg_dict['NDCG@10']:.4f}")
            print(f"Dynamic MRR@10:  {mrr_dict[mrr_key]:.4f}")
            print(f"Alpha stats:     mean = {mean_alpha:.4f}, std = {std_alpha:.4f}")

    return dynamic_results

dynamic_metrics = evaluate_dynamic_router(model, rd.test_retrieval_data, datasets_list)

In [ ]:
for i in results_ndcg.keys():
    print(f"{i} : {max(results_ndcg[i])}")

In [ ]:
for i in results_mrr.keys():
    print(f"{i} : {max(results_mrr[i])}")